# Day 4: Advanced Applications

**Applied Bayesian Statistics Workshop**

---

## Learning Objectives

1. Build a Gaussian mixture model in PyMC
2. Perform Bayesian A/B testing
3. Model simple time series with an autoregressive prior
4. Capstone: complete Bayesian analysis workflow from data to decision

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pymc as pm
import arviz as az
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12
az.style.use("arviz-darkgrid")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

---

## 1. Gaussian Mixture Model

A mixture of $K$ Gaussians models data that come from distinct subpopulations:

$$
p(y) = \sum_{k=1}^{K} w_k \, \mathcal{N}(y \mid \mu_k, \sigma_k)
$$

where $w_k$ are mixture weights summing to 1.

In [ ]:
# --- Generate data from a mixture of 3 Gaussians ---
K_true = 3
true_weights = np.array([0.35, 0.45, 0.20])
true_means = np.array([-3.0, 2.0, 7.0])
true_sds = np.array([1.0, 1.2, 0.8])

n_mix = 500
component = rng.choice(K_true, size=n_mix, p=true_weights)
y_mix = rng.normal(true_means[component], true_sds[component])

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_mix, bins=60, density=True, alpha=0.5, color="steelblue",
        edgecolor="white", label="Observed data")

x_grid = np.linspace(-8, 12, 500)
total_density = np.zeros_like(x_grid)
for k in range(K_true):
    comp_density = true_weights[k] * stats.norm.pdf(x_grid, true_means[k],
                                                     true_sds[k])
    ax.plot(x_grid, comp_density, "--", lw=1.5, label=f"Component {k+1}")
    total_density += comp_density

ax.plot(x_grid, total_density, "k-", lw=2.5, label="True mixture")
ax.set_xlabel("y")
ax.set_ylabel("Density")
ax.set_title("Synthetic Gaussian Mixture Data")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Fit Gaussian Mixture Model in PyMC ---
K = 3  # Assume we know the number of components

with pm.Model() as gmm:
    # Mixture weights
    w = pm.Dirichlet("w", a=np.ones(K))

    # Component means (ordered to prevent label switching)
    mu = pm.Normal("mu", mu=0, sigma=10, shape=K,
                   transform=pm.distributions.transforms.ordered,
                   initval=np.array([-3.0, 2.0, 7.0]))

    # Component standard deviations
    sigma = pm.HalfNormal("sigma", sigma=5, shape=K)

    # Mixture likelihood
    y_obs = pm.NormalMixture("y_obs", w=w, mu=mu, sigma=sigma,
                             observed=y_mix)

    trace_gmm = pm.sample(
        2000, tune=2000, cores=1, random_seed=RANDOM_SEED,
        target_accept=0.9, return_inferencedata=True
    )

print(az.summary(trace_gmm, var_names=["w", "mu", "sigma"]))

In [ ]:
# --- Plot fitted mixture vs data ---
w_post = az.extract(trace_gmm, var_names=["w"])["w"].values.mean(axis=1)
mu_post = az.extract(trace_gmm, var_names=["mu"])["mu"].values.mean(axis=1)
sigma_post = az.extract(trace_gmm, var_names=["sigma"])["sigma"].values.mean(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_mix, bins=60, density=True, alpha=0.4, color="steelblue",
        edgecolor="white", label="Data")

x_grid = np.linspace(-8, 12, 500)
fitted_density = np.zeros_like(x_grid)
for k in range(K):
    comp = w_post[k] * stats.norm.pdf(x_grid, mu_post[k], sigma_post[k])
    ax.plot(x_grid, comp, "--", lw=1.5,
            label=f"Comp {k+1}: w={w_post[k]:.2f}, "
                  f"mu={mu_post[k]:.1f}, sd={sigma_post[k]:.2f}")
    fitted_density += comp

ax.plot(x_grid, fitted_density, "k-", lw=2.5, label="Fitted mixture")
ax.set_xlabel("y")
ax.set_ylabel("Density")
ax.set_title("Gaussian Mixture Model: Posterior Fit")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---

## 2. Bayesian A/B Testing

We have two groups (A = control, B = treatment) and want to estimate
the probability that B is better than A, along with the magnitude of
the effect.

This is a direct application of posterior comparison -- no p-values needed.

In [ ]:
# --- Synthetic A/B test data (conversion rates) ---
n_A = 1200
n_B = 1100
true_rate_A = 0.12
true_rate_B = 0.15  # B is actually better

conversions_A = rng.binomial(n_A, true_rate_A)
conversions_B = rng.binomial(n_B, true_rate_B)

print(f"Group A: {conversions_A}/{n_A} conversions "
      f"({conversions_A/n_A:.1%})")
print(f"Group B: {conversions_B}/{n_B} conversions "
      f"({conversions_B/n_B:.1%})")

In [ ]:
# --- Bayesian A/B test model ---
with pm.Model() as ab_model:
    # Independent priors for each group's conversion rate
    p_A = pm.Beta("p_A", alpha=1, beta=1)
    p_B = pm.Beta("p_B", alpha=1, beta=1)

    # Derived quantities
    delta = pm.Deterministic("delta", p_B - p_A)
    relative_uplift = pm.Deterministic(
        "relative_uplift", (p_B - p_A) / p_A
    )

    # Likelihoods
    obs_A = pm.Binomial("obs_A", n=n_A, p=p_A, observed=conversions_A)
    obs_B = pm.Binomial("obs_B", n=n_B, p=p_B, observed=conversions_B)

    trace_ab = pm.sample(
        5000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print(az.summary(trace_ab, var_names=["p_A", "p_B", "delta",
                                       "relative_uplift"]))

In [ ]:
# --- Decision analysis ---
delta_samples = az.extract(trace_ab, var_names=["delta"])["delta"].values
uplift_samples = az.extract(trace_ab,
                             var_names=["relative_uplift"])["relative_uplift"].values

prob_B_better = (delta_samples > 0).mean()
prob_meaningful = (delta_samples > 0.01).mean()  # >1pp lift

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Posterior of delta
axes[0].hist(delta_samples, bins=80, density=True, alpha=0.6,
             color="steelblue", edgecolor="white")
axes[0].axvline(0, color="red", ls="--", lw=2, label="No difference")
axes[0].axvline(delta_samples.mean(), color="orange", lw=2,
                label=f"Mean: {delta_samples.mean():.4f}")
axes[0].set_xlabel(r"$\delta = p_B - p_A$")
axes[0].set_ylabel("Density")
axes[0].set_title(f"P(B > A) = {prob_B_better:.1%}")
axes[0].legend()

# Posterior of relative uplift
axes[1].hist(uplift_samples * 100, bins=80, density=True, alpha=0.6,
             color="mediumseagreen", edgecolor="white")
axes[1].axvline(0, color="red", ls="--", lw=2)
axes[1].set_xlabel("Relative Uplift (%)")
axes[1].set_ylabel("Density")
axes[1].set_title(f"Relative Uplift: "
                  f"{uplift_samples.mean()*100:.1f}% "
                  f"(94% HDI: [{np.percentile(uplift_samples*100, 3):.1f}%, "
                  f"{np.percentile(uplift_samples*100, 97):.1f}%])")

plt.tight_layout()
plt.show()

print(f"\nP(B is better than A):                  {prob_B_better:.1%}")
print(f"P(B is >1pp better than A):              {prob_meaningful:.1%}")
print(f"Expected uplift (absolute):              {delta_samples.mean():.4f}")
print(f"Expected uplift (relative):              {uplift_samples.mean()*100:.1f}%")

---

## 3. Simple Time Series with Autoregressive Prior

We model a time series $y_t$ where the underlying signal follows an AR(1) process:

$$
\mu_t = \phi \, \mu_{t-1} + \varepsilon_t^{\text{state}}, \quad \varepsilon_t^{\text{state}} \sim \mathcal{N}(0, \sigma_{\text{state}})
$$
$$
y_t \sim \mathcal{N}(\mu_t, \sigma_{\text{obs}})
$$

In [ ]:
# --- Generate synthetic AR(1) time series ---
T = 100
true_phi = 0.85
true_sigma_state = 0.5
true_sigma_obs = 1.0

# Generate latent states
mu_true = np.zeros(T)
for t in range(1, T):
    mu_true[t] = true_phi * mu_true[t-1] + rng.normal(0, true_sigma_state)

# Generate observations
y_ts = mu_true + rng.normal(0, true_sigma_obs, size=T)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(y_ts, "o-", ms=3, alpha=0.6, label="Observed $y_t$")
ax.plot(mu_true, "r-", lw=2, label=r"True $\mu_t$")
ax.set_xlabel("Time")
ax.set_ylabel("Value")
ax.set_title("Synthetic AR(1) Time Series")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Fit AR(1) model in PyMC ---
with pm.Model() as ar_model:
    # Priors
    phi = pm.Uniform("phi", lower=-1, upper=1)  # Stationarity
    sigma_state = pm.HalfNormal("sigma_state", sigma=2)
    sigma_obs = pm.HalfNormal("sigma_obs", sigma=2)

    # AR(1) latent process using a loop-free approach:
    # Build innovations, then cumulate with the AR coefficient
    innovations = pm.Normal("innovations", mu=0, sigma=sigma_state,
                            shape=T)

    # Construct the latent series via scan-like recursion
    # mu[0] = innovations[0]
    # mu[t] = phi * mu[t-1] + innovations[t]
    # We use the GaussianRandomWalk-inspired approach:
    mu_latent = pm.AR("mu_latent", rho=[phi], sigma=sigma_state,
                      shape=T, constant=False)

    # Observation likelihood
    y_obs = pm.Normal("y_obs", mu=mu_latent, sigma=sigma_obs,
                      observed=y_ts)

    trace_ar = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        target_accept=0.9, return_inferencedata=True
    )

print(az.summary(trace_ar, var_names=["phi", "sigma_state", "sigma_obs"]))

In [ ]:
# --- Plot recovered latent states ---
mu_post = az.extract(trace_ar, var_names=["mu_latent"])["mu_latent"].values
mu_post_mean = mu_post.mean(axis=1)
mu_post_lo = np.percentile(mu_post, 3, axis=1)
mu_post_hi = np.percentile(mu_post, 97, axis=1)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(y_ts, "o", ms=3, alpha=0.4, color="gray", label="Observed")
ax.plot(mu_true, "r-", lw=1.5, alpha=0.7, label=r"True $\mu_t$")
ax.plot(mu_post_mean, "b-", lw=2, label="Posterior mean")
ax.fill_between(range(T), mu_post_lo, mu_post_hi, alpha=0.2,
                color="steelblue", label="94% HDI")
ax.set_xlabel("Time")
ax.set_ylabel("Value")
ax.set_title("AR(1) Model: Recovered Latent States")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Posterior of phi ---
phi_samples = az.extract(trace_ar, var_names=["phi"])["phi"].values

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(phi_samples, bins=60, density=True, alpha=0.6, color="steelblue",
        edgecolor="white")
ax.axvline(true_phi, color="red", ls="--", lw=2,
           label=f"True: {true_phi}")
ax.axvline(phi_samples.mean(), color="orange", lw=2,
           label=f"Post. mean: {phi_samples.mean():.3f}")
ax.set_xlabel(r"$\phi$")
ax.set_ylabel("Density")
ax.set_title("Posterior of AR(1) Coefficient")
ax.legend()
plt.tight_layout()
plt.show()

---

## 4. Capstone: Complete Bayesian Analysis Workflow

We walk through a full analysis pipeline:

1. **Problem**: Does a new teaching method improve exam scores, controlling for prior ability?
2. **Data generation**: Simulate students across two conditions
3. **Model specification**: Bayesian regression with treatment effect
4. **Prior predictive check**
5. **Fitting**
6. **Diagnostics**
7. **Posterior predictive check**
8. **Inference and decision**

In [ ]:
# --- Step 1-2: Problem definition and data generation ---
n_students = 120
# 60 control, 60 treatment
treatment = np.array([0] * 60 + [1] * 60)

# Prior ability (continuous covariate, standardized)
prior_ability = rng.normal(0, 1, size=n_students)

# True model parameters
true_intercept = 65.0
true_ability_effect = 8.0
true_treatment_effect = 4.5  # The new method adds ~4.5 points
true_noise = 6.0

# Generate exam scores
exam_scores = (
    true_intercept
    + true_ability_effect * prior_ability
    + true_treatment_effect * treatment
    + rng.normal(0, true_noise, size=n_students)
)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for g, label, color in [(0, "Control", "coral"), (1, "Treatment", "steelblue")]:
    mask = treatment == g
    axes[0].scatter(prior_ability[mask], exam_scores[mask], alpha=0.5,
                    s=30, color=color, label=label)

axes[0].set_xlabel("Prior Ability (standardized)")
axes[0].set_ylabel("Exam Score")
axes[0].set_title("Exam Scores by Condition")
axes[0].legend()

axes[1].hist(exam_scores[treatment == 0], bins=20, alpha=0.5, color="coral",
             label="Control", density=True)
axes[1].hist(exam_scores[treatment == 1], bins=20, alpha=0.5,
             color="steelblue", label="Treatment", density=True)
axes[1].set_xlabel("Exam Score")
axes[1].set_ylabel("Density")
axes[1].set_title("Score Distributions")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Control mean:   {exam_scores[treatment == 0].mean():.1f}")
print(f"Treatment mean: {exam_scores[treatment == 1].mean():.1f}")
print(f"Raw difference: {exam_scores[treatment == 1].mean() - exam_scores[treatment == 0].mean():.1f}")

In [ ]:
# --- Step 3: Model specification ---
# y_i = beta_0 + beta_ability * ability_i + beta_trt * treatment_i + eps_i

with pm.Model() as capstone_model:
    # Priors -- weakly informative
    beta_0 = pm.Normal("beta_0", mu=60, sigma=20)
    beta_ability = pm.Normal("beta_ability", mu=0, sigma=10)
    beta_trt = pm.Normal("beta_trt", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=15)

    # Expected value
    mu = beta_0 + beta_ability * prior_ability + beta_trt * treatment

    # Likelihood
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=exam_scores)

print("Model specified.")

In [ ]:
# --- Step 4: Prior predictive check ---
with capstone_model:
    prior_pred = pm.sample_prior_predictive(
        samples=500, random_seed=RANDOM_SEED
    )

fig, ax = plt.subplots(figsize=(10, 5))
# Plot a few prior predictive datasets
prior_y = prior_pred.prior_predictive["y_obs"].values[0]  # chain 0
for i in range(min(50, prior_y.shape[0])):
    ax.hist(prior_y[i], bins=30, alpha=0.03, color="steelblue",
            density=True)

ax.hist(exam_scores, bins=30, density=True, alpha=0.7, color="coral",
        edgecolor="white", label="Observed data")
ax.set_xlabel("Exam Score")
ax.set_ylabel("Density")
ax.set_title("Prior Predictive Check (blue=simulated, red=observed)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 5: Fit the model ---
with capstone_model:
    trace_capstone = pm.sample(
        3000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print(az.summary(trace_capstone,
                 var_names=["beta_0", "beta_ability", "beta_trt", "sigma"]))

In [ ]:
# --- Step 6: Diagnostics ---
az.plot_trace(trace_capstone,
              var_names=["beta_0", "beta_ability", "beta_trt", "sigma"])
plt.tight_layout()
plt.show()

# Check diagnostics
summary = az.summary(trace_capstone,
                     var_names=["beta_0", "beta_ability", "beta_trt", "sigma"])
print("\n--- Convergence Check ---")
print(f"All R-hat < 1.01: {(summary['r_hat'] < 1.01).all()}")
print(f"All ESS_bulk > 400: {(summary['ess_bulk'] > 400).all()}")
div = int(trace_capstone.sample_stats["diverging"].sum().values)
print(f"Divergences: {div}")

In [ ]:
# --- Step 7: Posterior predictive check ---
with capstone_model:
    ppc_capstone = pm.sample_posterior_predictive(
        trace_capstone, random_seed=RANDOM_SEED
    )

az.plot_ppc(ppc_capstone, num_pp_samples=100)
plt.title("Posterior Predictive Check: Capstone Model")
plt.tight_layout()
plt.show()

In [ ]:
# --- Step 8: Inference and decision ---
beta_trt_samples = az.extract(
    trace_capstone, var_names=["beta_trt"]
)["beta_trt"].values

prob_positive = (beta_trt_samples > 0).mean()
prob_meaningful = (beta_trt_samples > 2).mean()  # >2 points is "meaningful"

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(beta_trt_samples, bins=60, density=True, alpha=0.6,
        color="steelblue", edgecolor="white")
ax.axvline(0, color="red", ls="--", lw=2, label="No effect")
ax.axvline(2, color="orange", ls="--", lw=1.5,
           label="Minimal meaningful effect (2 pts)")
ax.axvline(true_treatment_effect, color="green", ls="-", lw=2,
           label=f"True effect ({true_treatment_effect})")
ax.axvline(beta_trt_samples.mean(), color="navy", lw=2,
           label=f"Posterior mean: {beta_trt_samples.mean():.2f}")

# HDI
hdi = az.hdi(beta_trt_samples, hdi_prob=0.94)
ax.axvspan(hdi[0], hdi[1], alpha=0.15, color="steelblue",
           label=f"94% HDI: [{hdi[0]:.1f}, {hdi[1]:.1f}]")

ax.set_xlabel("Treatment Effect (points)")
ax.set_ylabel("Density")
ax.set_title("Posterior of Treatment Effect")
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()

print("\n" + "=" * 55)
print("           DECISION SUMMARY")
print("=" * 55)
print(f"Posterior mean treatment effect:  {beta_trt_samples.mean():.2f} points")
print(f"94% HDI:                         [{hdi[0]:.2f}, {hdi[1]:.2f}]")
print(f"P(effect > 0):                   {prob_positive:.1%}")
print(f"P(effect > 2 points):            {prob_meaningful:.1%}")
print(f"\nConclusion: There is a {prob_positive:.0%} probability that the")
print(f"new teaching method improves scores, with an expected")
print(f"improvement of {beta_trt_samples.mean():.1f} points.")
print("=" * 55)

---

## Workshop Summary

Over four days, we have covered:

| Day | Topics |
|-----|--------|
| 1 | Bayes' theorem, conjugate models, prior predictive checks, PyMC basics, Bayesian linear regression |
| 2 | MCMC (Metropolis-Hastings, NUTS), diagnostics, posterior predictive checks, logistic and Poisson regression |
| 3 | Hierarchical models, shrinkage, non-centered parameterization, model comparison (WAIC/LOO) |
| 4 | Mixture models, A/B testing, time series, complete Bayesian workflow |

### Recommended Resources

- *Statistical Rethinking* (Richard McElreath) -- excellent conceptual introduction
- *Bayesian Data Analysis* (Gelman et al.) -- comprehensive reference
- PyMC documentation: https://www.pymc.io/
- ArviZ documentation: https://python.arviz.org/

### The Bayesian Workflow Checklist

1. Define the question and the generative model
2. Choose priors (weakly informative by default)
3. Prior predictive check: do simulated data look reasonable?
4. Fit the model (MCMC sampling)
5. Check diagnostics: R-hat, ESS, divergences, trace plots
6. Posterior predictive check: does the model reproduce observed data patterns?
7. Interpret posteriors: HDIs, probabilities of hypotheses, effect sizes
8. Compare models if needed (LOO, WAIC)
9. Communicate results with full uncertainty